In [ ]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths_, datasets


In [ ]:
# specify target FPS
TARGET_FPS = 60

# left wrist, right wrist, left thigh, right thigh, head, pelvis
vi_mask = torch.tensor([1961, 5424, 876, 4362, 411, 3021])
ji_mask = torch.tensor([18, 19, 1, 2, 15, 0])
body_model = ParametricModel(paths_.smpl_file)


In [ ]:


# def _syn_acc(v, smooth_n=4):
#     """Synthesize accelerations from vertex positions."""
#     mid = smooth_n // 2
#     scale_factor = TARGET_FPS ** 2 #

#     acc = torch.stack([(v[i] + v[i + 2] - 2 * v[i + 1]) * scale_factor for i in range(0, v.shape[0] - 2)])
#     acc = torch.cat((torch.zeros_like(acc[:1]), acc, torch.zeros_like(acc[:1])))

#     if mid != 0:
#         acc[smooth_n:-smooth_n] = torch.stack(
#             [(v[i] + v[i + smooth_n * 2] - 2 * v[i + smooth_n]) * scale_factor / smooth_n ** 2
#              for i in range(0, v.shape[0] - smooth_n * 2)])
#     return acc

# def process_amass():
#     def _foot_ground_probs(joint):
#         """Compute foot-ground contact probabilities."""
#         dist_lfeet = torch.norm(joint[1:, 10] - joint[:-1, 10], dim=1)
#         dist_rfeet = torch.norm(joint[1:, 11] - joint[:-1, 11], dim=1)
#         lfoot_contact = (dist_lfeet < 0.008).int()
#         rfoot_contact = (dist_rfeet < 0.008).int()
#         lfoot_contact = torch.cat((torch.zeros(1, dtype=torch.int), lfoot_contact))
#         rfoot_contact = torch.cat((torch.zeros(1, dtype=torch.int), rfoot_contact))
#         return torch.stack((lfoot_contact, rfoot_contact), dim=1)

#     # enable skipping processed files
#     try:
#         processed = [fpath.name for fpath in (paths_.processed_datasets).iterdir()]
#     except FileNotFoundError:
#         processed = []

#     for ds_name in datasets.amass_datasets:
#         # skip processed 
#         if f"{ds_name}.pt" in processed:
#             continue

#         data_pose, data_trans, data_beta, length = [], [], [], []
#         print("\rReading", ds_name)

#         for npz_fname in tqdm(sorted(glob.glob(os.path.join(paths_.raw_amass, ds_name, "*/*_poses.npz")))):
#             try: cdata = np.load(npz_fname)
#             except: continue

#             framerate = int(cdata['mocap_framerate'])
#             if framerate not in [120, 60, 59]:
#                 continue

#             # enable downsampling
#             step = max(1, round(framerate / TARGET_FPS))

#             # print(f"\tProcessing {npz_fname} with original FPS {framerate} and step {step}")
#             data_pose.extend(cdata['poses'][::step].astype(np.float32))
#             data_trans.extend(cdata['trans'][::step].astype(np.float32))
#             data_beta.append(cdata['betas'][:10])
#             length.append(cdata['poses'][::step].shape[0])

#         if len(data_pose) == 0:
#             print(f"AMASS dataset, {ds_name} not supported")
#             continue

#         length = torch.tensor(length, dtype=torch.int)
#         print(f"Total frames in {ds_name}: {length}")
#         shape = torch.tensor(np.asarray(data_beta, np.float32))
#         tran = torch.tensor(np.asarray(data_trans, np.float32))
#         pose = torch.tensor(np.asarray(data_pose, np.float32)).view(-1, 52, 3)

#         # include the left and right index fingers in the pose
#         pose[:, 23] = pose[:, 37]     # right hand 
#         pose = pose[:, :24].clone()   # only use body + right and left fingers

#         # align AMASS global frame with DIP
#         amass_rot = torch.tensor([[[1, 0, 0], [0, 0, 1], [0, -1, 0.]]])
#         tran = amass_rot.matmul(tran.unsqueeze(-1)).view_as(tran)
#         pose[:, 0] = math.rotation_matrix_to_axis_angle(
#             amass_rot.matmul(math.axis_angle_to_rotation_matrix(pose[:, 0])))

#         print("Synthesizing IMU accelerations and orientations")
#         b = 0
#         out_pose, out_shape, out_tran, out_joint, out_vrot, out_vacc, out_contact = [], [], [], [], [], [], []
#         for i, l in tqdm(list(enumerate(length))):
#             if l <= 12: b += l; print("\tdiscard one sequence with length", l); continue
#             p = math.axis_angle_to_rotation_matrix(pose[b:b + l]).view(-1, 24, 3, 3)
#             grot, joint, vert = body_model.forward_kinematics(p, shape[i], tran[b:b + l], calc_mesh=True)

#             out_pose.append(p.clone())  # N, 24, 3, 3
#             out_tran.append(tran[b:b + l].clone())  # N, 3
#             out_shape.append(shape[i].clone())  # 10
#             out_joint.append(joint[:, :24].contiguous().clone())  # N, 24, 3
#             out_vacc.append(_syn_acc(vert[:, vi_mask]))  # N, 6, 3
#             out_contact.append(_foot_ground_probs(joint).clone()) # N, 2

#             out_vrot.append(grot[:, ji_mask])  # N, 6, 3, 3
#             b += l

#         print("Saving...")
#         data = {
#             'joint': out_joint,
#             'pose': out_pose,
#             'shape': out_shape,
#             'tran': out_tran,
#             'acc': out_vacc,
#             'ori': out_vrot,
#             'contact': out_contact
#         }
#         data_path = paths_.processed_datasets / f"{ds_name}.pt"
#         torch.save(data, data_path)
#         print(f"Synthetic AMASS dataset is saved at: {data_path}")

# def create_directories():
#     paths_.processed_datasets.mkdir(exist_ok=True, parents=True)
#     paths_.eval_dir.mkdir(exist_ok=True, parents=True)



In [ ]:
# create_directories()

In [ ]:
# process_amass()

In [1]:
#libraries


import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 
import copy

import articulate as art
from config import *
from utils import *
from helpers import *


import os
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
from articulate import model as art_model
from utils.model_utils import reduced_pose_to_full
from articulate.math import r6d_to_rotation_matrix
from articulate.evaluator import MeanPerJointErrorEvaluator
from articulate.math import RotationRepresentation


class PoseDataset(Dataset):
    def __init__(self, fold: str='train', evaluate: str=None, finetune: str=None):
        super().__init__()
        self.fold = fold
        self.evaluate = evaluate
        self.finetune = finetune
        self.bodymodel = art.model.ParametricModel(paths_.smpl_file)
        self.combos = list(amass.combos.items())
        self.data = self._prepare_dataset()

    def _get_data_files(self, data_folder):
        if self.fold == 'train':
            return self._get_train_files(data_folder)
        elif self.fold == 'test':
            return self._get_test_files()
        else:
            raise ValueError(f"Unknown data fold: {self.fold}.")

    def _get_train_files(self, data_folder):
        if self.finetune:
            return [datasets.finetune_datasets[self.finetune]]
        else:
            return [x.name for x in data_folder.iterdir() if not x.is_dir()]

    def _get_test_files(self):
        return [datasets.test_datasets[self.evaluate]]

    def _prepare_dataset(self):
        data_folder = paths_.processed_datasets / ('eval' if (self.finetune or self.evaluate) else '')
        data_files = self._get_data_files(data_folder)

        print(f"\n{'='*60}")
        print(f"Loading datasets for {self.fold.upper()} mode")
        print(f"Datasets: {data_files}")
        print(f"{'='*60}\n")

        data = {key: [] for key in [
            'imu_inputs', 'pose_outputs', 'joint_outputs',
            'tran_outputs', 'vel_outputs', 'foot_outputs'
        ]}

        for data_file in tqdm(data_files):
            try:
                file_data = torch.load(data_folder / data_file)
                self._process_file_data(file_data, data)
            except Exception as e:
                print(f"Error processing {data_file}: {e}")

        return data

    def _process_file_data(self, file_data, data):
        accs, oris, poses, trans = file_data['acc'], file_data['ori'], file_data['pose'], file_data['tran']
        joints = file_data.get('joint', [None] * len(poses))
        foots = file_data.get('contact', [None] * len(poses))

        for acc, ori, pose, tran, joint, foot in zip(accs, oris, poses, trans, joints, foots):

            acc, ori = acc[:, :5]/amass.acc_scale, ori[:, :5]

            pose_global, joint = self.bodymodel.forward_kinematics(
                pose=pose.view(-1, 216)
            )

            pose = pose if self.evaluate else pose_global.view(-1, 24, 3, 3)
            joint = joint.view(-1, 24, 3)


            self._process_combo_data(acc, ori, pose, joint, tran, foot, data)

    def _pad_to_length(self, tensor, target_len):
        """Pad tensor with zeros along dim 0 to reach target_len."""
        if tensor.shape[0] < target_len:
            pad_size = target_len - tensor.shape[0]
            padding = torch.zeros(pad_size, *tensor.shape[1:], dtype=tensor.dtype)
            tensor = torch.cat([tensor, padding], dim=0)
        return tensor

    def _process_combo_data(self, acc, ori, pose, joint, tran, foot, data):

        for combo_name, c in self.combos:


            combo_acc = torch.zeros_like(acc)
            combo_ori = torch.zeros_like(ori)
            combo_acc[:, c] = acc[:, c]
            combo_ori[:, c] = ori[:, c]

            imu_input = torch.cat([combo_acc.flatten(1), combo_ori.flatten(1)], dim=1)

            data_len = len(imu_input) if self.evaluate else datasets.window_length


            for key, value in zip(
                ['imu_inputs', 'pose_outputs', 'joint_outputs', 'tran_outputs'],
                [imu_input, pose, joint, tran]
            ):
                splits = torch.split(value, data_len)

                # pad last split with zeros if shorter than window length
                splits = list(splits)
                if splits[-1].shape[0] < data_len:
                    splits[-1] = self._pad_to_length(splits[-1], data_len)

                data[key].extend(splits)

            if not (self.evaluate or self.finetune):
                self._process_translation_data(joint, tran, foot, data_len, data)

    def _process_translation_data(self, joint, tran, foot, data_len, data):

        root_vel = torch.cat((torch.zeros(1, 3), tran[1:] - tran[:-1]))
        vel = torch.cat((torch.zeros(1, 24, 3), torch.diff(joint, dim=0)))
        vel[:, 0] = root_vel

        vel = vel * (datasets.fps / amass.vel_scale)

        vel_splits = list(torch.split(vel, data_len))
        foot_splits = list(torch.split(foot, data_len))

        # pad last splits with zeros if shorter than window length
        if vel_splits[-1].shape[0] < data_len:
            vel_splits[-1] = self._pad_to_length(vel_splits[-1], data_len)
        if foot_splits[-1].shape[0] < data_len:
            foot_splits[-1] = self._pad_to_length(foot_splits[-1], data_len)

        data['vel_outputs'].extend(vel_splits)
        data['foot_outputs'].extend(foot_splits)

    def __getitem__(self, idx):

        imu = self.data['imu_inputs'][idx].float()
        joint = self.data['joint_outputs'][idx].float()
        tran = self.data['tran_outputs'][idx].float()

        num_pred_joints = len(amass.pred_joints_set)

        pose = art.math.rotation_matrix_to_r6d(
            self.data['pose_outputs'][idx]
        ).reshape(-1, num_pred_joints, 6)[:, amass.pred_joints_set] \
         .reshape(-1, 6*num_pred_joints)


        if self.evaluate or self.finetune:
            return imu, pose, joint, tran

        vel = self.data['vel_outputs'][idx].float()
        contact = self.data['foot_outputs'][idx].float()

        return imu, pose, joint, tran, vel, contact

    def __len__(self):
        return len(self.data['imu_inputs'])
    




def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)


class PoseDataModule(L.LightningDataModule):
    def __init__(self, finetune: str = None):
        super().__init__()
        self.finetune = finetune
        self.hypers = finetune_hypers if self.finetune else train_hypers

    def setup(self, stage: str):
        if stage == 'fit':
            dataset = PoseDataset(fold='train', finetune=self.finetune)
            train_size = int(0.9 * len(dataset))
            val_size = len(dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        elif stage == 'test':
            self.test_dataset = PoseDataset(fold='test', finetune=self.finetune)

    def _dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.hypers.batch_size, 
            collate_fn=pad_seq, 
            num_workers=0, #self.hypers.num_workers, 
            shuffle=True, 
            drop_last=True
        )

    def train_dataloader(self):
        return self._dataloader(self.train_dataset)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def test_dataloader(self):
        return self._dataloader(self.test_dataset)

In [ ]:

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class SinusoidalTimestepEmbedding(nn.Module):
    """Sinusoidal embedding for diffusion timestep, projected through an MLP."""
    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(),
            nn.Linear(d_model * 4, d_model),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """t: (B,) integer timesteps -> (B, d_model)"""
        half = self.d_model // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.d_model % 2 == 1:
            emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
        return self.mlp(emb)


# ---------------------------------------------------------------------------
# Diffusion noise schedule helpers
# ---------------------------------------------------------------------------
def linear_beta_schedule(num_timesteps: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    return torch.linspace(beta_start, beta_end, num_timesteps)


def cosine_beta_schedule(num_timesteps: int, s: float = 0.008):
    steps = torch.arange(num_timesteps + 1, dtype=torch.float64)
    alpha_bar = torch.cos(((steps / num_timesteps) + s) / (1 + s) * (math.pi / 2)) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return torch.clamp(betas, 0.0001, 0.9999).float()


class GaussianDiffusion(nn.Module):
    """Manages the forward (noising) and reverse (denoising) diffusion process."""
    def __init__(self, num_timesteps: int = 1000, schedule: str = "cosine"):
        super().__init__()
        self.num_timesteps = num_timesteps

        if schedule == "cosine":
            betas = cosine_beta_schedule(num_timesteps)
        else:
            betas = linear_beta_schedule(num_timesteps)

        alphas = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])

        # Register all as buffers so they move with .to(device)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alpha_bar", alpha_bar)
        self.register_buffer("alpha_bar_prev", alpha_bar_prev)
        self.register_buffer("sqrt_alpha_bar", torch.sqrt(alpha_bar))
        self.register_buffer("sqrt_one_minus_alpha_bar", torch.sqrt(1.0 - alpha_bar))
        self.register_buffer("sqrt_recip_alpha", torch.sqrt(1.0 / alphas))
        self.register_buffer(
            "posterior_variance",
            betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar),
        )

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None):
        """Forward diffusion: q(x_t | x_0)."""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = self.sqrt_alpha_bar[t]
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t]
        # broadcast to (B, 1, 1) if x0 is (B, T, D)
        while sqrt_ab.dim() < x0.dim():
            sqrt_ab = sqrt_ab.unsqueeze(-1)
            sqrt_omab = sqrt_omab.unsqueeze(-1)
        return sqrt_ab * x0 + sqrt_omab * noise, noise

    @torch.no_grad()
    def p_sample(self, model, x_t, t_index: int, cond=None):
        """Single reverse step: p(x_{t-1} | x_t)."""
        B = x_t.shape[0]
        t_tensor = torch.full((B,), t_index, device=x_t.device, dtype=torch.long)
        noise_pred = model(x_t, t_tensor, cond=cond)
        beta = self.betas[t_index]
        sqrt_recip_alpha = self.sqrt_recip_alpha[t_index]
        sqrt_omab = self.sqrt_one_minus_alpha_bar[t_index]
        mean = sqrt_recip_alpha * (x_t - beta / sqrt_omab * noise_pred)
        if t_index > 0:
            sigma = torch.sqrt(self.posterior_variance[t_index])
            mean = mean + sigma * torch.randn_like(x_t)
        return mean

    @torch.no_grad()
    def ddim_sample(self, model, x_T, num_steps: int = 50, eta: float = 0.0, cond=None):
        """DDIM sampling for faster, higher-quality generation.

        Args:
            model: noise prediction network
            x_T: starting noise (B, T, D)
            num_steps: number of DDIM steps (subset of full schedule)
            eta: stochasticity (0 = deterministic DDIM, 1 = DDPM)
            cond: optional conditioning input
        """
        step_size = max(1, self.num_timesteps // num_steps)
        timesteps = list(range(0, self.num_timesteps, step_size))[::-1]

        x = x_T
        for i, t_cur in enumerate(timesteps):
            B = x.shape[0]
            t_tensor = torch.full((B,), t_cur, device=x.device, dtype=torch.long)
            noise_pred = model(x, t_tensor, cond=cond)

            alpha_bar_t = self.alpha_bar[t_cur]
            t_prev = timesteps[i + 1] if i + 1 < len(timesteps) else 0
            alpha_bar_prev = self.alpha_bar[t_prev] if t_prev > 0 else torch.tensor(1.0, device=x.device)

            # predicted x0
            x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)
            x0_pred = x0_pred.clamp(-5, 5)

            sigma = eta * torch.sqrt(
                (1 - alpha_bar_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_prev)
            )
            dir_xt = torch.sqrt(torch.clamp(1 - alpha_bar_prev - sigma ** 2, min=0)) * noise_pred

            x = torch.sqrt(alpha_bar_prev) * x0_pred + dir_xt
            if t_cur > 0 and eta > 0:
                x = x + sigma * torch.randn_like(x)

        return x

    @torch.no_grad()
    def sample(self, model, shape, device, cond=None):
        """Full reverse sampling loop: x_T -> x_0."""
        x = torch.randn(shape, device=device)
        for t in reversed(range(self.num_timesteps)):
            x = self.p_sample(model, x, t, cond=cond)
        return x


class TemporalTransformerDiffusion(nn.Module):
    """IMU-conditioned diffusion model (noise predictor) for pose + translation.

    Uses cross-attention (TransformerDecoder) to condition on IMU sensor inputs,
    dramatically improving accuracy over the unconditional baseline.
    """
    def __init__(
        self,
        pose_dim: int,
        tran_dim: int = 3,
        imu_dim: int = 60,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 6,
        dim_feedforward: int = 1024,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.pose_dim = pose_dim
        self.tran_dim = tran_dim
        combined_dim = pose_dim + tran_dim

        # Noisy input projection (2-layer MLP)
        self.in_proj = nn.Sequential(
            nn.Linear(combined_dim, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )

        # Timestep embedding
        self.time_emb = SinusoidalTimestepEmbedding(d_model)

        # Positional encoding (shared)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        # IMU conditioning encoder
        self.imu_proj = nn.Sequential(
            nn.Linear(imu_dim, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )
        imu_enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.imu_encoder = nn.TransformerEncoder(imu_enc_layer, num_layers=2)

        # Decoder with self-attention + cross-attention to IMU
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # LayerNorm before output heads
        self.out_norm = nn.LayerNorm(d_model)

        # Separate output heads for pose and translation noise
        self.pose_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, pose_dim),
        )
        self.tran_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.SiLU(),
            nn.Linear(d_model // 2, tran_dim),
        )

    def forward(self, x_t: torch.Tensor, t: torch.Tensor, cond: torch.Tensor = None):
        """Predict noise from noisy input + timestep + optional IMU conditioning.

        x_t  : (B, T, pose_dim + tran_dim)  noisy concatenated pose+tran
        t    : (B,) integer diffusion timesteps
        cond : (B, T, imu_dim) IMU sensor input (optional)
        Returns: (B, T, pose_dim + tran_dim) predicted noise
        """
        h = self.in_proj(x_t)                         # (B, T, d_model)
        t_emb = self.time_emb(t).unsqueeze(1)          # (B, 1, d_model)
        h = h + t_emb                                  # broadcast add timestep info
        h = self.pos_enc(h)

        if cond is not None:
            # Encode IMU conditioning
            c = self.imu_proj(cond)                     # (B, T, d_model)
            c = self.pos_enc(c)
            c = self.imu_encoder(c)                     # (B, T, d_model)
            # Decode with cross-attention to IMU
            h = self.decoder(h, c)                      # (B, T, d_model)
        else:
            # Fallback: self-attention only (no conditioning)
            h = self.decoder(h, h)

        h = self.out_norm(h)
        pose_noise = self.pose_head(h)
        tran_noise = self.tran_head(h)
        return torch.cat([pose_noise, tran_noise], dim=-1)


class EMA:
    """Exponential Moving Average of model parameters for smoother outputs."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for s_param, m_param in zip(self.shadow.parameters(), model.parameters()):
                s_param.data.mul_(self.decay).add_(m_param.data, alpha=1 - self.decay)

    
def validate(val_loader, model, diffusion, device):
    model.eval()
    val_loss_sum = 0.0
    val_total_frames = 0

    with torch.no_grad():
        for batch in val_loader:
            (inputs, input_lengths), (outputs, output_lengths) = batch

            imu_cond = inputs.to(device)
            pose = outputs["poses"].to(device)
            tran = outputs["trans"].to(device)
            lengths = torch.as_tensor(output_lengths["poses"], device=device)

            B, T, F_pose = pose.shape
            x0 = torch.cat([pose, tran], dim=-1)  # (B, T, pose_dim + tran_dim)

            # sample random timesteps
            t = torch.randint(0, diffusion.num_timesteps, (B,), device=device)
            noise = torch.randn_like(x0)
            x_t, _ = diffusion.q_sample(x0, t, noise)

            noise_pred = model(x_t, t, cond=imu_cond)

            loss_matrix = nn.functional.mse_loss(noise_pred, noise, reduction="none")

            mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
            frame_mask = mask.unsqueeze(-1).float()

            masked_loss = (loss_matrix * frame_mask).sum()
            num_valid = frame_mask.sum()

            val_loss_sum += masked_loss.item()
            val_total_frames += num_valid.item()

    val_loss = val_loss_sum / val_total_frames if val_total_frames > 0 else 0.0
    print(f"Validation Loss: {val_loss:.6f}")
    return val_loss


def training(train_loader, val_loader, model, diffusion, num_epochs=1, device=None, patience: int = 10, min_delta: float = 1e-4):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    diffusion = diffusion.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-2,
    )

    # Cosine annealing LR scheduler with warmup
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Exponential Moving Average
    ema = EMA(model, decay=0.9999)

    criterion = nn.MSELoss(reduction="none")

    best_val_loss = None
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        model.train()
        epoch_weighted_loss = 0.0
        total_frames = 0

        for batch in train_loader:
            (inputs, input_lengths), (outputs, output_lengths) = batch

            imu_cond = inputs.to(device)
            pose = outputs["poses"].to(device)
            tran = outputs["trans"].to(device)
            lengths = torch.as_tensor(output_lengths["poses"], device=device)

            B, T, F_pose = pose.shape

            # concatenate pose + tran as the clean signal x_0
            x0 = torch.cat([pose, tran], dim=-1)  # (B, T, pose_dim + tran_dim)

            # sample random diffusion timesteps per sample
            t = torch.randint(0, diffusion.num_timesteps, (B,), device=device)

            # forward diffusion: add noise according to schedule
            noise = torch.randn_like(x0)
            x_t, _ = diffusion.q_sample(x0, t, noise)

            # predict the noise (conditioned on IMU)
            # noise_pred = model(x_t, t, cond=imu_cond)
            noise_pred = model(x_t, t, cond=None)

            loss_matrix = criterion(noise_pred, noise)

            # mask padding frames
            mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
            frame_mask = mask.unsqueeze(-1).float()

            masked_loss = (loss_matrix * frame_mask).sum()
            num_valid = frame_mask.sum()

            if num_valid > 0:
                loss = masked_loss / num_valid
            else:
                loss = torch.tensor(0.0, device=device, requires_grad=True)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Update EMA after each optimizer step
            ema.update(model)

            with torch.no_grad():
                grad_norm = sum(
                    p.grad.data.norm(2).item() ** 2
                    for p in model.parameters() if p.grad is not None
                ) ** 0.5

            print(f"Batch Loss: {loss.item():.6f} | Grad Norm: {grad_norm:.4f}")

            epoch_weighted_loss += masked_loss.item()
            total_frames += num_valid.item()

        scheduler.step()

        epoch_loss = epoch_weighted_loss / total_frames if total_frames > 0 else 0.0
        current_lr = optimizer.param_groups[0]['lr']
        print("===================================")
        print(f"Epoch {epoch}")
        print(f"Epoch Loss: {epoch_loss:.6f} | LR: {current_lr:.2e}")
        print("===================================")

        # Validate using EMA model
        val_loss = validate(val_loader, ema.shadow, diffusion, device)

        # Early stopping based on val loss
        if best_val_loss is None or val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(ema.shadow.state_dict(), "diffusion_model_cond_best_step_3.pth")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve}/{patience} epochs.")
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

    torch.save(ema.shadow.state_dict(), "diffusion_model_cond_final_step_3.pth")


# ---------------------------------------------------------------------------
# Sampling helper (call after training)
# ---------------------------------------------------------------------------
@torch.no_grad()
def generate_samples(model, diffusion, num_samples: int, seq_len: int, pose_dim: int, tran_dim: int, device, cond=None):
    """Generate pose + translation sequences from noise, optionally conditioned on IMU."""
    model.eval()
    shape = (num_samples, seq_len, pose_dim + tran_dim)
    x_T = torch.randn(shape, device=device)
    samples = diffusion.ddim_sample(model, x_T, num_steps=50, eta=0.0, cond=cond)
    pose_samples = samples[..., :pose_dim]
    tran_samples = samples[..., pose_dim:]
    return pose_samples, tran_samples






In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
datamodule = PoseDataModule(finetune=None)
datamodule.setup(stage='fit')

train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()

POSE_DIM = 144
TRAN_DIM = 3
NUM_DIFFUSION_STEPS = 1000

diffusion = GaussianDiffusion(
    num_timesteps=NUM_DIFFUSION_STEPS,
    schedule="cosine",
).to(device)

model = TemporalTransformerDiffusion(
    pose_dim=POSE_DIM,
    tran_dim=TRAN_DIM,
    imu_dim=60,
    d_model=256,
    nhead=8,
    num_layers=6,
    dim_feedforward=1024,
    dropout=0.1,
).to(device)

print(f"IMU-Conditioned Diffusion Model initialized — pose_dim={POSE_DIM}, tran_dim={TRAN_DIM}, "
        f"timesteps={NUM_DIFFUSION_STEPS}")

training(
    train_loader=train_loader,
    val_loader=val_loader,
    model=model,
    diffusion=diffusion,
    num_epochs=100,
    patience=10,
    device=device,
)



Loading datasets for TRAIN mode
Datasets: ['ACCAD.pt', 'BioMotionLab_NTroje.pt', 'BMLhandball.pt', 'BMLmovi.pt', 'CMU.pt', 'DanceDB.pt', 'DFaust_67.pt', 'Eyes_Japan_Dataset.pt', 'HUMAN4D.pt', 'HumanEva.pt', 'MPI_HDM05.pt', 'MPI_Limits.pt', 'MPI_mosh.pt', 'SFU.pt', 'SSM_synced.pt', 'TCD_handMocap.pt', 'TotalCapture.pt', 'Transitions_mocap.pt']



  0%|          | 0/18 [00:00<?, ?it/s]C:\Users\degu03\AppData\Local\Temp\ipykernel_18312\1341353039.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.lo

IMU-Conditioned Diffusion Model initialized — pose_dim=144, tran_dim=3, timesteps=1000
Batch Loss: 152.620132 | Grad Norm: 1.0000
Batch Loss: 151.495621 | Grad Norm: 1.0000
Batch Loss: 150.692688 | Grad Norm: 1.0000
Batch Loss: 150.019135 | Grad Norm: 1.0000
Batch Loss: 149.585526 | Grad Norm: 1.0000
Batch Loss: 149.211212 | Grad Norm: 1.0000
Batch Loss: 148.703720 | Grad Norm: 1.0000
Batch Loss: 148.604355 | Grad Norm: 1.0000
Batch Loss: 148.431839 | Grad Norm: 1.0000
Batch Loss: 148.069901 | Grad Norm: 1.0000
Batch Loss: 148.229828 | Grad Norm: 1.0000
Batch Loss: 148.053406 | Grad Norm: 1.0000
Batch Loss: 147.998474 | Grad Norm: 1.0000
Batch Loss: 147.803253 | Grad Norm: 1.0000
Batch Loss: 147.841858 | Grad Norm: 1.0000
Batch Loss: 147.691162 | Grad Norm: 1.0000
Batch Loss: 147.691162 | Grad Norm: 1.0000
Batch Loss: 147.683029 | Grad Norm: 1.0000
Batch Loss: 147.535156 | Grad Norm: 1.0000
Batch Loss: 147.492371 | Grad Norm: 1.0000
Batch Loss: 147.708908 | Grad Norm: 1.0000
Batch Loss